In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

c:\07_Python_Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

In [3]:
FreeOpenRouterModels=["google/gemma-4-26b-a4b-it:free","google/gemma-4-31b-it:free", "openai/gpt-oss-120b:free", "openai/gpt-oss-20b:free"]
PaidOpenRouterModels=["openai/gpt-oss-safeguard-20b","openai/gpt-oss-120b","google/gemma-3-12b-it","openai/gpt-4.1-nano"]

In [5]:
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openrouter.chat.completions.create(model=PaidOpenRouterModels[3], messages=messages)
    return response.choices[0].message.content

In [28]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Markdown(label="Response:") ## Without borders,
models=gr.Dropdown(label="Select a model", choices=FreeOpenRouterModels, value=PaidOpenRouterModels[3], info="Select a model to use for the response")

c:\07_Python_Projects\.venv\Lib\site-packages\gradio\components\dropdown.py:235: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: openai/gpt-4.1-nano or set allow_custom_value=True.
  warnings.warn(


In [ ]:
view=gr.Interface(
    fn=message_gpt, 
    title="Get your answers from LLM", 
    inputs=[message_input], 
    outputs=[message_output], 
    flagging_mode="never"
    )

view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [18]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),
# Please take a look at the Intermediate Python guide in the guides folder

def stream_gpt(prompt, model=PaidOpenRouterModels[3]):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openrouter.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [27]:
view=gr.Interface(
    fn=stream_gpt, 
    title="Get your answers from LLM", 
    inputs=[message_input,models], 
    outputs=[message_output], 
    flagging_mode="never",
    examples=[["Explain the Transformer architecture to a layperson","openai/gpt-oss-safeguard-20b"],
    ["Explain the Transformer architecture to an aspiring AI engineer","google/gemma-3-12b-it"]]
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## Offline Models


In [1]:
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

c:\07_Python_Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [5]:
OllamaModels=["deepseek-r1:1.5b",
"llama3.2:latest",
"codeguru:latest",
"codellama:latest",
"qwen3.5:0.8b",
"mistral:latest",
"llama2:latest",
"gemma:2b"
]

In [9]:
system_message="You are a Helpful Assistant"

In [10]:

def stream_gpt(prompt, model=OllamaModels[3]):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [7]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Markdown(label="Response:") ## Without borders,
models=gr.Dropdown(label="Select a model", choices=OllamaModels, value=OllamaModels[3], info="Select a model to use for the response")

In [11]:
view=gr.Interface(
    fn=stream_gpt, 
    title="Get your answers from LLM", 
    inputs=[message_input,models], 
    outputs=[message_output], 
    flagging_mode="never",
    examples=[["Explain the Transformer architecture to a layperson","gemma:2b"],
    ["Explain the Transformer architecture to an aspiring AI engineer","codellama:latest"]]
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
